# 36 — Experience Parsing
**Goal:** Extract company, role, duration, and responsibilities.

Experience is the most information-dense section of a resume and the least regular: header lines vary ("Google, Mountain View — Senior Data Scientist"), dates come in every format, and the real payload is an arbitrary list of bullets. This chapter builds a line-driven state machine that groups those lines into experience entries.

**Why it matters for resumes / ATS:** experience is what recruiters actually evaluate — role progression, tenure, and quantified achievements. Parsing it into `(company, role, duration, bullets)` is what makes role-level matching ("has this candidate been a Senior Data Scientist?") and tenure scoring possible, and it feeds the bullet scoring in Ch. 37.

## 1. Experience Pattern Recognition

Before writing a parser, study the shape of the data. Experience entries follow a near-universal template: a **header line** (`Company, Location — Role`), a **date range** line, then **bullet points** of achievements — and the whole pattern repeats per employer.

**What the code does:** sets up `exp_text`, a two-company sample (Google and Amazon), and prints the expected shape: header, date range, bullets.

**Why it matters:** every design decision in the parser comes from this template. The header line carries the em-dash delimiter (`—`) between location and role; dates are anchored on month names or 4-digit years; bullets start with `-`, `•`, or whitespace. If you can name these regularities, you can encode them — and you know which resumes (non-standard layouts) will defeat the parser.

In [ ]:
exp_text = """Google, Mountain View — Senior Data Scientist
Jan 2020 - Present
- Developed NLP pipelines processing 10M+ documents daily
- Led team of 5 ML engineers
- Reduced model latency by 40%

Amazon, Seattle — Data Scientist II
2018 - 2020
- Built recommendation systems
- Improved CTR by 25%
"""

print("Experience entries typically follow:")
print("  Company, Location — Role")
print("  Date range (start - end)")
print("  Bullet points of achievements")

## 2. Experience Parser

`parse_experience()` is a small **state machine**: it walks lines, and each line either starts a new entry, sets the duration, or appends a bullet — depending on which pattern it matches.

**What the code does:** three checks per line, in order:
- **Header regex** `^([A-Za-z\s.]+),?\s*([A-Za-z\s]+)?\s*[—\-–]\s*(.+)$` — on a match, the current entry is finalized and a new `{"company", "role", "duration", "bullets"}` starts.
- **Date regex** (month names or `\d{4}` followed by `-`) — records the first 40 characters of the line as `duration`.
- **Bullet strip** `^[\s•\-*–]+` — any other non-trivial line (length > 10) is added to the current entry's bullets.

**Verified on the sample:** two entries come out — `Google | Senior Data Scientist | Jan 2020 - Present` with 3 bullets, and `Amazon | Data Scientist II | 2018 - 2020` with 2 bullets — each bullet's leading dash stripped.

**Try it:** the header regex requires the dash delimiter; a resume that writes "Google, Senior Data Scientist" on one line without a dash will fail to start a new entry — a known coverage gap.

In [ ]:
import re

def parse_experience(text):
    """Extract experience entries."""
    entries = []
    lines = text.split("\n")
    current = None
    
    for line in lines:
        ls = line.strip()
        if not ls: continue
        
        # Company line: "Company, Location — Role"
        company_match = re.match(r"^([A-Za-z\s.]+),?\s*([A-Za-z\s]+)?\s*[—\-–]\s*(.+)$", ls)
        if company_match:
            if current: entries.append(current)
            current = {"company": company_match.group(1).strip(), 
                       "role": company_match.group(3).strip(), "duration": "", "bullets": []}
            continue
        
        # Date line
        date_match = re.search(r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|\d{4})\s*\d{0,4}\s*-", ls)
        if date_match and current:
            current["duration"] = ls[:40]
            continue
        
        # Bullet point
        bullet = re.sub(r"^[\s•\-*–]+", "", ls)
        if bullet and current and len(bullet) > 10:
            current["bullets"].append(bullet)
    
    if current: entries.append(current)
    return entries

entries = parse_experience(exp_text)
for e in entries:
    print(f"\n  {e['company']:15s} | {e['role']:25s} | {e['duration'][:20]}")
    for b in e['bullets']:
        print(f"    - {b[:50]}...")

## 3. Duration Calculation

Tenure is a recruiter filter ("5+ years of experience"), but durations arrive as free text: "5+ years", "2020 - Present", "2018 - 2020". `parse_duration()` converts them to a single comparable number — years.

**What the code does:** three fallbacks, in order:
- an explicit `N years` pattern (optional `+`) → `N`;
- a date-range pattern that finds all 4-digit years and subtracts the first from the last;
- otherwise `0`.

**Verified on the sample:** `"5+ years"` → `5`, `"2018 - 2020"` → `2`, `"3 years"` → `3` — but `"2020 - Present"` → `0`, because "Present" is not a year. That is the honest limitation: active roles undercount tenure until you special-case "Present" against a reference date such as today.

In [ ]:
from datetime import datetime

def parse_duration(duration_str):
    """Calculate years from a duration string."""
    years_match = re.search(r"(\d+)\s*(?:\+)?\s*years?", duration_str, re.IGNORECASE)
    if years_match: return int(years_match.group(1))
    
    # Try date range
    range_match = re.findall(r"\b(\d{4})\b", duration_str)
    if len(range_match) >= 2:
        return int(range_match[-1]) - int(range_match[0])
    return 0

for d in ["5+ years", "2020 - Present", "2018 - 2020", "3 years"]:
    print(f"  '{d}' -> {parse_duration(d)} years")

## Summary: Pattern matching extracts structured experience. Duration calc estimates tenure.

**Experience parsing is a state machine over lines: header, date, bullet — repeat.**

The three-pattern loop is deliberately simple and fully explainable: it recovers company, role, duration, and bullets without any model, and it degrades predictably (a missing dash, an unparsable date) rather than failing silently. Duration math is an estimate, not truth — "Present" needs a reference date to be counted.

The output shape — `{company, role, duration, bullets}` — is the `Experience` model from Ch. 39, and the bullets it collects are the raw material Ch. 37 scores for STAR compliance next.